# Relatório Final do Trabalho (RFT) - Sistema de Processamento Visual
**Disciplina:** Processamento Digital de Imagens (MCZA018)  
**Equipe 9**

**Eduardo de Souza Carrilho - RA: 11201812084**
**Gabriel Figueiredo de Souza - RA: 11202230332**

# 1. Introdução

### 1.1 Objetivos do Trabalho
O objetivo central deste trabalho é o desenvolvimento de um Sistema de Processamento Visual (SPV) interativo para a automação da correção de gabaritos de múltipla escolha. Utilizando Python e OpenCV, o projeto visa:
* Aplicar conceitos de processamento de vídeo em tempo real com feedback visual imediato ao usuário.
* Implementar um pipeline robusto de PDI que execute desde a detecção de marcas de registro até a extração de atributos e cálculo de notas.
* Desenvolver uma interface interativa que permita o "travamento" do alvo e a correção "on-the-fly".

### 1.2 Cenário de Aplicação (CA)
O cenário de aplicação foca no ambiente escolar cotidiano, especificamente para professores que lidam com um alto volume de avaliações. Diferente de sistemas industriais que exigem hardware caro, esta solução é voltada para a "realidade da sala de aula", permitindo que o docente utilize uma webcam simples para digitalizar e corrigir provas instantaneamente.

**Benefícios do Sistema:**
* **Otimização do Tempo:** Redução do esforço manual de correção, eliminando erros por fadiga visual.
* **Interatividade:** O sistema guia o professor através de alertas na tela ("Alinhe o gabarito" / "Alvo Travado"), facilitando o uso por leigos.
* **Digitalização Eficiente:** Transformação de dados físicos em notas digitais prontas para lançamento em sistemas acadêmicos.

### 1.3 Fundamentação Teórica
A robustez do código desenvolvido baseia-se nos seguintes conceitos técnicos:

#### A. Realce e Binarização Adaptativa (CLAHE)
Para lidar com variações de iluminação no laboratório, o sistema utiliza o **CLAHE** (Contrast Limited Adaptive Histogram Equalization), que melhora o contraste local. A binarização é feita via **Threshold Adaptativo**, permitindo que o sistema isole as marcações mesmo em frames com sombras ou iluminação não uniforme.

#### B. Detecção de Marcas de Registro e Homografia
Diferente de uma simples detecção de bordas, o código implementa a busca por **marcas de registro** (quadrados pretos nos cantos). Através da função `getPerspectiveTransform`, o sistema calcula a matriz de homografia para retificar a imagem, garantindo que o gabarito seja analisado sempre em uma geometria fixa (600x800), independente da inclinação da folha.

#### C. Processamento Morfológico e Análise de Contornos
Após a retificação, o sistema utiliza **Operadores Morfológicos de Abertura** para limpar ruídos. A identificação das alternativas (bolhas) é feita através da filtragem de contornos por área e proporção (aspect ratio), garantindo que apenas círculos com dimensões compatíveis sejam processados.

#### D. Segmentação e Watershed
Para situações onde as marcações do aluno podem tocar as bordas do círculo ou entre si, o código está estruturado para aplicar o algoritmo **Watershed**. Esta técnica trata a imagem como um mapa topográfico, utilizando a transformada de distância para encontrar os centros das bolhas e separá-las com precisão cirúrgica, evitando contagens errôneas de píxeis.


#### E. Extração de Atributos (Lógica de Preenchimento)
A decisão final sobre qual alternativa foi marcada baseia-se na contagem de píxeis não-zero em cada máscara circular. O sistema compara a densidade de preenchimento contra um **limiar crítico**, permitindo identificar questões corretas, erradas, em branco ou anuladas (múltiplas marcações).

## 2. Materiais e Métodos

### 2.1 Diagrama de Blocos Funcional do SPV
O sistema foi desenhado através de um *pipeline* sequencial de processamento de imagem, operando em ciclo (*loop* contínuo) a cada *frame* capturado pela câmara:

1. **Aquisição de Vídeo:** Captura do *frame* BGR (`cv2.VideoCapture`).
2. **Pré-processamento e Deteção de Alvo:** Conversão para tons de cinzento, suavização e limiarização.
3. **Alinhamento (Warp Perspective):** Deteção de quatro marcas de registo (quadrados) e cálculo da matriz de homografia.
4. **Isolamento da Região de Interesse (ROI):** Recorte e redimensionamento da folha para dimensões fixas (600x800 píxeis).
5. **Segmentação e Morfologia:** Aplicação de *CLAHE*, Binarização Adaptativa Invertida e Abertura Morfológica (`cv2.morphologyEx`).
6. **Extração de Atributos:** Deteção dos contornos das bolhas, ordenação espacial (linha a linha, da esquerda para a direita) e cálculo da densidade de píxeis não-zero (brancos).
7. **Decisão e Output:** Comparação da densidade com o limiar crítico (1150 píxeis), cálculo da nota final face ao gabarito matriz e exibição no ecrã.

### 2.2 Descrição da Implementação do SPV
O software foi desenvolvido na linguagem Python (versão 3.10+), recorrendo à API do OpenCV (`cv2`) para o processamento visual e ao `numpy` para a manipulação de matrizes matemáticas. A implementação está modularizada nas seguintes funções principais:

* **`align_gabarito` e `four_point_transform`:** O sistema não assume que a folha ocupa todo o ecrã. Em vez disso, procura os contornos externos com proporção (`aspect ratio`) quadrada e área específica, identificando as quatro "marcas de registo" nos cantos do gabarito. Através da função `cv2.getPerspectiveTransform`, a imagem é achatada e planificada, removendo qualquer distorção de ângulo causada pela postura do utilizador.
* **`preprocess_image`:** Aplica o algoritmo CLAHE (`cv2.createCLAHE`) para equalizar o histograma localmente, corrigindo sombras e gradientes de luz. De seguida, utiliza o `cv2.adaptiveThreshold` com vizinhança Gaussiana para garantir que a binarização não se perde em zonas mal iluminadas do papel.
* **`apply_watershed`:** Implementado como técnica avançada de segmentação, o algoritmo de *Watershed* é capaz de calcular a transformada de distância (`cv2.distanceTransform`) para encontrar os picos de intensidade, separando bolhas adjacentes que o aluno possa ter preenchido com demasiada tinta.
* **`avaliar_respostas`:** Esta é a função de extração de informação. Filtra as 50 bolhas pelas suas proporções circulares (0.7 a 1.3 de *aspect ratio*). Organiza-as numa matriz espacial e cria uma máscara para cada alternativa. Se a contagem (`cv2.countNonZero`) ultrapassar o limiar de 1150 píxeis, a alternativa é considerada marcada. A função está programada para lidar com falhas de preenchimento, assinalando "BRANCO" (sem marcações) ou "ANULADA" (múltiplas marcações).

### 2.3 Lista de Diretórios
O projeto está organizado numa estrutura de diretórios que separa os recursos de dados, o código-fonte e as documentações auxiliares. Os diretórios que compõem o repositório são:

* **Raiz do Projeto:**
    * `README.md`: Documentação principal do repositório com instruções de configuração e visão geral.
    * `Roteiro de Laboratório.pdf`: Documento contendo o guia passo a passo utilizado pelos voluntários no Teste de Campo.
* **Diretório `/src` (Source):**
    * `spv.ipynb`: O núcleo do projeto. Contém o código Python, a implementação do pipeline OpenCV e as análises de dados no formato Jupyter Notebook.
* **Diretório `/data/images`:**
    * `gabarito_teste.png`: Imagem padrão do gabarito utilizada para validar a deteção de bolhas e o alinhamento de perspetiva.

### 2.4 Análise Técnica e Desempenho
O sistema desenvolvido atende na íntegra ao cenário de aplicação escolar (EdTech) proposto. A interatividade foi assegurada através de mecanismos de *feedback* visual (mensagens de "Alvo Travado" ou alertas de erro como "Leu 48/50 bolhas").

**Métricas Objetivas (Numéricas):**
* **Velocidade de Processamento:** O sistema processa o *frame* congelado (pela tecla 'C') num tempo impercetível ao utilizador (fração de segundo), providenciando o resultado instantaneamente (Operação *On-the-fly*).
* **Acurácia do Alinhamento:** O rácio de fixação das marcas de registo é próximo de 100% em ambientes com iluminação frontal homogénea e papel desdobrado.
* **Robustez da Segmentação:** A calibração morfológica garante que áreas inferiores a 500 píxeis quadrados ou que não se assemelhem a círculos sejam descartadas, prevenindo falsos positivos devidos a rabiscos ou poeiras no papel.

**Métricas Qualitativas e Conclusões:**
Apesar da alta eficácia demonstrada, o sistema exige que a folha esteja orientada com uma determinada margem de tolerância rotacional (o utilizador não pode colocar o papel de pernas para o ar, uma vez que as coordenadas lógicas seriam invertidas). Adicionalmente, verificou-se que visões laterais excessivamente agudas (com grandes distorções ao nível da perspetiva de captura) afetam a proporção das marcas de registo, impedindo o "travamento" do alvo. 
Conclui-se que o uso do CLAHE mitigou as limitações associadas à binarização global perante o surgimento de sombras.

## 3. Laboratório Experimental (LEx)

### 3.1 Roteiro do Laboratório Experimental
Abaixo é apresentado o guia passo a passo fornecido aos utilizadores leigos durante o Teste de Campo.

**1. Introdução ao Sistema**
Bem-vindo(a) ao experimento prático do nosso Corretor Automático de Gabaritos! Este sistema foi desenvolvido para facilitar a vida de professores e educadores, automatizando a correção de provas de múltipla escolha utilizando apenas uma webcam comum e técnicas de visão computacional.
Como o sistema funciona para você (Usuário):
* **Câmera em Tempo Real:** Ao iniciar, o sistema abrirá uma janela mostrando a imagem da sua webcam.
* **Busca de Âncoras:** O sistema procurará por 4 quadrados pretos nos cantos do gabarito. Se ele não os achar, mostrará a mensagem em vermelho: "Alinhe o gabarito na tela...".
* **Travamento do Alvo:** Quando o sistema reconhecer a folha corretamente, ele desenhará uma borda verde ao redor da prova e mostrará a mensagem: "ALVO TRAVADO! Aperte 'C' para Corrigir".
* **Correção:** Ao apertar 'C', o sistema congela a imagem, "achata" a folha virtualmente para corrigir a perspectiva, lê as marcações feitas a caneta e exibe o resultado final na tela, indicando os acertos em verde, erros em vermelho, e questões rasuradas em roxo, além de calcular a sua nota.

**2. Preparação e Materiais Necessários**
* O gabarito oficial impresso (gerado pelo próprio sistema).
* Uma caneta preta ou azul escura (esferográfica comum ou canetinha).
* O computador com o sistema já iniciado pelo instrutor (ambiente Jupyter).

**3. Procedimento Experimental**
* **Experimento A: O Caminho Feliz (Condições Ideais)**
  1. Pegue um gabarito impresso novo.
  2. Preencha as bolhas de pelo menos 5 questões de forma correta (pinte completamente).
  3. Posicione o papel na frente da webcam, garantindo que os 4 cantos pretos estejam visíveis.
  4. Quando a tela mostrar "ALVO TRAVADO!", aperte a tecla C.
  5. Ação: Observe a janela de resultado. O sistema computou a nota corretamente?
  6. Aperte R para reiniciar o scanner.
* **Experimento B: Tratamento de Erros Comuns**
  1. Em uma questão nova, pinte DUAS alternativas (simulando um aluno que errou e tentou rasurar). Deixe uma questão totalmente em BRANCO.
  2. Posicione o papel na frente da câmera e aperte C quando travar o alvo.
  3. Ação: Verifique se o sistema identificou a dupla marcação com a palavra "ANULADA" e a questão vazia com a palavra "BRANCO".
  4. Aperte R para reiniciar.
* **Experimento C: Robustez e Ângulo**
  1. Segure o gabarito preenchido levemente inclinado (na diagonal) em relação à câmera, mas ainda garantindo que os 4 cantos apareçam.
  2. Ação: Observe se o quadrado verde do sistema consegue acompanhar a inclinação do papel. Aperte C.
  3. Verifique no resultado final se o programa conseguiu "desentortar" a imagem da prova antes de corrigir.

**4. Questionário Didático e Enquete Subjetiva**
O roteiro incluiu perguntas para verificar o entendimento do utilizador (ex: "O que o sistema exige que esteja visível na câmera para liberar a tecla 'C'?") e uma enquete de satisfação estruturada em escala Likert (1 a 5) e questões dissertativas qualitativas.

---

### 3.2 Análise Quantitativa (Médias de Usabilidade)
O sistema foi avaliado por 8 participantes através de critérios de usabilidade (escala 1 a 5). Abaixo, apresentam-se as médias obtidas:

| Critério de Avaliação | Média (1-5) |
| :--- | :---: |
| Gostaria de usar esse sistema com frequência | **4,75** |
| Achei o sistema desnecessariamente complexo | **1,00** |
| Achei o sistema fácil de usar | **5,00** |
| Precisaria de suporte técnico para usar | **1,12** |
| Funções do sistema bem integradas | **5,00** |
| Inconsistências no sistema | **1,00** |
| Aprendizagem rápida por outras pessoas | **5,00** |
| Achei o sistema complicado de usar | **1,25** |
| Senti-me confiante ao usar o sistema | **5,00** |
| Precisou aprender muita coisa antes de usar | **1,00** |
| **O sistema é interativo?** | **5,00** |

### 3.3 Análise Qualitativa Consolidada
* **Destaques Positivos:** A capacidade de entender a perspectiva da folha e o funcionamento com papéis inclinados foram os pontos mais fortes. A interatividade e o feedback visual ("Alvo Travado") garantiram segurança aos usuários.
* **Pontos de Melhoria:** Identificou-se a necessidade de o gabarito estar na orientação vertical. Sugeriu-se o uso de padrões de âncoras que permitam detecção de rotação 360° e melhoria na leitura em ângulos laterais extremos.
---

## 4. Conclusões
Os objetivos propostos na introdução foram plenamente atingidos. A equipa conseguiu desenvolver um Sistema de Processamento Visual "on-the-fly" robusto, substituindo a correção manual e monótona de gabaritos físicos por um *pipeline* digital automatizado (EdTech).

**Pontos Positivos:**
* A implementação combinada de **CLAHE** com **Binarização Adaptativa** mitigou fortemente o problema crónico de sombras sobre o papel.
* A Transformação de Perspetiva baseada em "marcas de registo" (em vez das margens do papel) revelou-se um sucesso absoluto na estabilidade do *software*, permitindo a correção mesmo com folhas amachucadas ou rasgadas nas bordas.
* O sistema lida elegantemente com exceções do mundo real (rasuras e questões em branco).

**Pontos Negativos e Limitações:**
* A limitação primária reside na sensibilidade à orientação. O utilizador é obrigado a manter o papel na vertical correta.
* O limite de processamento da homografia falha se a inclinação da câmara for demasiado paralela à mesa (visão rasante).

**Conclusão Final:**
A modelagem do trabalho provou-se altamente funcional. O projeto integra de forma prática e criativa a ementa de Processamento Digital de Imagens (filtragens morfológicas, transformações geométricas e segmentação Watershed), resultando numa solução de alto valor utilitário.

---

## 5. Referências Bibliográficas
1. GONZALEZ, R. C.; WOODS, R. E. **Processamento Digital de Imagens**. 4. ed. São Paulo: Pearson Education do Brasil, 2018.
2. OpenCV. **Image Thresholding & Geometric Transformations**. Documentação Oficial da API. Disponível em: <https://docs.opencv.org/4.x/>.
3. ZAMPIROLLI, F.; KURASHIMA, C. **Especificação de Projeto e Notas de Aula de MCZA018**. Universidade Federal do ABC (UFABC), 2026.

---

## 6. Anexos
### A - Dados Brutos do Laboratório Experimental (LEx)

Este anexo apresenta as respostas individuais coletadas durante a aplicação do teste de campo com 8 usuários voluntários. As perguntas foram baseadas no questionário SUS (*System Usability Scale*) e em questões dissertativas de desempenho.

#### 1. Estrutura do Questionário (Escala 1 a 5)
As perguntas quantitativas referem-se aos seguintes critérios:
* **Q1:** Gostaria de usar esse sistema com frequência.
* **Q2:** Achei o sistema desnecessariamente complexo.
* **Q3:** Achei o sistema fácil de usar.
* **Q4:** Precisaria de suporte técnico para usar o sistema.
* **Q5:** Achei que as funções foram bem integradas.
* **Q6:** Achei que havia inconsistências no sistema.
* **Q7:** Pessoas aprenderiam a usar rapidamente.
* **Q8:** Achei o sistema complicado de usar.
* **Q9:** Senti-me confiante ao usar o sistema.
* **Q10:** Precise aprender muitas coisas antes de usar.
* **Q11:** Você achou o sistema interativo?

#### 2. Tabela de Respostas Quantitativas

| Usuário | Q1 | Q2 | Q3 | Q4 | Q5 | Q6 | Q7 | Q8 | Q9 | Q10 | Q11 |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |
| 2 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |
| 3 | 3 | 1 | 5 | 2 | 5 | 1 | 5 | 2 | 5 | 1 | 5 |
| 4 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |
| 5 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |
| 6 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |
| 7 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |
| 8 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 | 1 | 5 |

#### 3. Respostas Dissertativas (Qualitativas)

| Usuário | Do que mais gostou? | Pontos de Melhoria / Sugestões | Resultados dos Experimentos |
|:---:|:---|:---|:---|
| **1** | Capacidade de entender a perspectiva. | N/A | Corrigiu a prova corretamente. |
| **2** | O quão completo o sistema é. | Gostei de todo o sistema. | Obtive os resultados da correção. |
| **3** | Facilidade de uso. | A obrigatoriedade do template. | Como esperado, com resultados corretos. |
| **4** | Intuitivo e profissional. | N/A | Marcações OK aprovadas e NOK descartadas. |
| **5** | Praticidade na correção. | Dificuldade inicial de alinhamento / Funcionar invertido. | Verificou corretamente todos os testes. |
| **6** | Facilidade para acessar funções. | N/A | Preciso, exceto em visão lateral extrema. |
| **7** | Funciona com folhas inclinadas. | Gabarito precisa estar em pé (orientação). | Todos os testes funcionaram corretamente. |
| **8** | Facilidade de usar. | N/A | Marquei as questões e o sistema corrigiu. |

---
*Dados coletados em laboratório de campo para a disciplina de PDI.*
### B - Códigos-fonte
Abaixo encontra-se o código fonte gerado durante a realização desse projeto:


In [ ]:
import cv2
import numpy as np
import datetime
import os

In [ ]:
def preprocess_image(frame):
    """Aplica CLAHE e Binarização Adaptativa com janela ampla para evitar donuts."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    equalized = clahe.apply(gray)
    
    blurred = cv2.GaussianBlur(equalized, (5, 5), 0)
    
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 71, 10)
                                   
    kernel = np.ones((3,3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    
    return thresh

In [ ]:
def apply_watershed(thresh_roi, original_roi):
    """
    Aplica o algoritmo Watershed para separar bolhas que possam ter sido
    pintadas juntas pelo aluno.
    """
    kernel = np.ones((3,3), np.uint8)
    sure_bg = cv2.dilate(thresh_roi, kernel, iterations=2)
    
    dist_transform = cv2.distanceTransform(thresh_roi, cv2.DIST_L2, 5)
    ret, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    ret, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    markers = cv2.watershed(original_roi, markers)
    original_roi[markers == -1] = [0, 0, 255]
    
    return original_roi, markers

In [ ]:
def order_points(pts):
    """Ordena as coordenadas: [sup-esq, sup-dir, inf-dir, inf-esq]."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

In [ ]:
def four_point_transform(image, pts):
    """Achata e recorta a região da folha para um tamanho FIXO."""
    rect = order_points(pts)
    
    LARGURA_FIXA = 600
    ALTURA_FIXA = 800

    dst = np.array([
        [0, 0],
        [LARGURA_FIXA - 1, 0],
        [LARGURA_FIXA - 1, ALTURA_FIXA - 1],
        [0, ALTURA_FIXA - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (LARGURA_FIXA, ALTURA_FIXA))
    
    return warped

In [ ]:
def align_gabarito(frame):
    """Busca os 4 quadrados pretos nos cantos (marcas de registro) para alinhar."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 11, 2)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    marcadores = []
    
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.04 * peri, True)
        area = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        
        if h == 0: continue
        proporcao = float(w) / h
        
        if len(approx) == 4 and 150 < area < 5000 and 0.8 <= proporcao <= 1.2:
            M = cv2.moments(c)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                marcadores.append([cX, cY])
                
    if len(marcadores) == 4:
        pts = np.array(marcadores, dtype="float32")
        rect = order_points(pts)
        
        LARGURA_FIXA = 600
        ALTURA_FIXA = 800
        
        dst = np.array([
            [0, 0],
            [LARGURA_FIXA - 1, 0],
            [LARGURA_FIXA - 1, ALTURA_FIXA - 1],
            [0, ALTURA_FIXA - 1]], dtype="float32")

        M = cv2.getPerspectiveTransform(rect, dst)
        gabarito_alinhado = cv2.warpPerspective(frame, M, (LARGURA_FIXA, ALTURA_FIXA))
        
        contorno_folha = rect.reshape((-1, 1, 2)).astype(np.int32)
        
        return gabarito_alinhado, True, contorno_folha
    
    return frame, False, None

In [ ]:
def sort_contours_top_to_bottom(contours):
    bounding_boxes = [cv2.boundingRect(c) for c in contours]
    (contours, bounding_boxes) = zip(*sorted(zip(contours, bounding_boxes),
                                             key=lambda b: b[1][1], reverse=False))
    return list(contours)

In [ ]:
def order_bubbles_left_to_right(contours):
    bounding_boxes = [cv2.boundingRect(c) for c in contours]
    (contours, bounding_boxes) = zip(*sorted(zip(contours, bounding_boxes),
                                             key=lambda b: b[1][0], reverse=False))
    return list(contours)

In [ ]:
def avaliar_respostas(thresh_img, color_img, bolhas_contours, gabarito_oficial):
    """Lê os pixels de cada bolha e compara com o gabarito oficial."""
    bolhas_contours = sort_contours_top_to_bottom(bolhas_contours)
    
    acertos = 0
    total_questoes = len(gabarito_oficial)
    
    LIMIAR_PREENCHIMENTO = 1150
    
    for q, i in enumerate(range(0, len(bolhas_contours), 5)):
        if q >= total_questoes: break
        
        linha_bolhas = bolhas_contours[i:i+5]
        linha_bolhas = order_bubbles_left_to_right(linha_bolhas)
        
        pixels_marcados = []
        for c in linha_bolhas:
            mask = np.zeros(thresh_img.shape, dtype="uint8")
            cv2.drawContours(mask, [c], -1, 255, -1)
            mask = cv2.bitwise_and(thresh_img, thresh_img, mask=mask)
            total_pixels = cv2.countNonZero(mask)
            pixels_marcados.append(total_pixels)
                        
        marcadas = [idx for idx, p in enumerate(pixels_marcados) if p > LIMIAR_PREENCHIMENTO]
        
        x_status, y_status = cv2.boundingRect(linha_bolhas[-1])[:2]
        
        resposta_certa = gabarito_oficial[q]
        indice_certo = ['A', 'B', 'C', 'D', 'E'].index(resposta_certa)
        
        if len(marcadas) == 0:
            cv2.putText(color_img, "BRANCO", (x_status + 40, y_status + 15), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.drawContours(color_img, [linha_bolhas[indice_certo]], -1, (255, 150, 0), 2)
            
        elif len(marcadas) > 1:
            # Questão Anulada (Múltiplas marcações)
            cv2.putText(color_img, "ANULADA", (x_status + 40, y_status + 15), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
            for idx in marcadas:
                cv2.drawContours(color_img, [linha_bolhas[idx]], -1, (255, 0, 255), 3)
                
        else:
            indice_marcado = marcadas[0]
            resposta_aluno = ['A', 'B', 'C', 'D', 'E'][indice_marcado]
            
            if resposta_aluno == resposta_certa:
                acertos += 1
                cv2.drawContours(color_img, [linha_bolhas[indice_marcado]], -1, (0, 255, 0), 3)
            else:
                cv2.drawContours(color_img, [linha_bolhas[indice_marcado]], -1, (0, 0, 255), 3)
                cv2.drawContours(color_img, [linha_bolhas[indice_certo]], -1, (0, 255, 0), 3)

    nota = (acertos / total_questoes) * 10
    cv2.putText(color_img, f"NOTA: {nota:.1f} / 10.0", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)
    
    return color_img, nota

In [ ]:
def main(video_input=0):
    """
    Função principal. video_input pode ser 0 (webcam) ou o caminho para um arquivo .mp4
    """
    cap = cv2.VideoCapture(video_input)
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0 

    GABARITO_MATRIZ = ['A', 'B', 'C', 'D', 'E', 'D', 'C', 'B', 'A', 'B']
    nome_janela = "SPV - Equipe 9"
    
    if not cap.isOpened():
        print("Erro: Não foi possível abrir o vídeo/webcam.")
        return

    print("Pressione 'q' para sair.")

    modo_foto = False
    frame_alinhado_salvo = None
    
    while True:
        ret, frame = cap.read()
        if not ret: 
            break
        
        frame_camera = frame.copy()
        frame_gravacao = frame.copy()
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
            
        if not modo_foto:
            frame_alinhado, sucesso, contorno_folha = align_gabarito(frame)
            
            if sucesso:
                cv2.drawContours(frame_camera, [contorno_folha], -1, (0, 255, 0), 4)
                cv2.putText(frame_camera, "ALVO TRAVADO! Aperte 'C' para Corrigir", (20, 40), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
                if key == ord('c'):
                    modo_foto = True
                    frame_alinhado_salvo = frame_alinhado.copy()
            else:
                cv2.putText(frame_camera, "Alinhe o gabarito na tela...", (20, 40), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            cv2.imshow(nome_janela, frame_camera)
            
        else:
            thresh = preprocess_image(frame_alinhado_salvo)

            cv2.imshow("DEBUG - Visao Binaria", thresh)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            bolhas_encontradas = []
            for c in contours:
                x, y, w, h = cv2.boundingRect(c)
                proporcao = w / float(h)
                area = cv2.contourArea(c)
                
                if 0.7 <= proporcao <= 1.3 and 500 < area < 2500:
                    bolhas_encontradas.append(c)
                    
            resultado_visual = frame_alinhado_salvo.copy()
            
            if len(bolhas_encontradas) == 50:
                resultado_visual, nota = avaliar_respostas(thresh, resultado_visual, bolhas_encontradas, GABARITO_MATRIZ)
                
                cv2.putText(resultado_visual, "Aperte 'R' para escanear outra prova", (20, 780), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            else:
                cv2.putText(resultado_visual, f"ERRO: Leu {len(bolhas_encontradas)}/50 bolhas.", (20, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(resultado_visual, "Aperte 'R' e tente tirar a foto novamente.", (20, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            cv2.imshow("Scanner Corretor", resultado_visual)
            
            cv2.putText(frame_camera, "CORRECAO CONCLUIDA NA OUTRA JANELA", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
            cv2.imshow(nome_janela, frame_camera)
            
            
            if key == ord('r'):
                modo_foto = False
                cv2.destroyWindow("Scanner Corretor")

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
def gerar_gabarito(filename="../data/images/gabarito_teste.png"):
    largura, altura = 800, 1100
    img = np.ones((altura, largura, 3), dtype=np.uint8) * 255
    margem, tamanho_quadrado = 50, 40

    cv2.rectangle(img, (margem, margem), (margem+tamanho_quadrado, margem+tamanho_quadrado), (0,0,0), -1)
    cv2.rectangle(img, (largura-margem-tamanho_quadrado, margem), (largura-margem, margem+tamanho_quadrado), (0,0,0), -1)
    cv2.rectangle(img, (margem, altura-margem-tamanho_quadrado), (margem+tamanho_quadrado, altura-margem), (0,0,0), -1)
    cv2.rectangle(img, (largura-margem-tamanho_quadrado, altura-margem-tamanho_quadrado), (largura-margem, altura-margem), (0,0,0), -1)

    cv2.putText(img, "Gabarito - Sistema de Processamento Visual", (100, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)

    inicio_y, inicio_x, espacamento_y, espacamento_x, raio = 250, 180, 65, 80, 22
    opcoes = ['A', 'B', 'C', 'D', 'E']

    for q in range(1, 11):
        y = inicio_y + (q-1) * espacamento_y
        cv2.putText(img, f"{q:02d}.", (inicio_x - 80, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)
        for i, letra in enumerate(opcoes):
            x = inicio_x + i * espacamento_x
            cv2.circle(img, (x, y), raio, (0,0,0), 2)
            cv2.putText(img, letra, (x - 10, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 2)

    cv2.imwrite(filename, img)
    print(f"Imagem de teste salva como {filename}")

In [ ]:
if __name__ == "__main__":    
    main(0)

In [ ]:
gerar_gabarito()